# 🌲 Stage 2: Random Forest on ARIMA Residuals — Hybrid Multi-Echelon Forecasting
### Thesis: Hybrid Time Series & ML Approach for Multi-Echelon Supply Chain Demand Forecasting

This notebook completes the **hybrid model**. It loads the residual file produced by the ARIMA notebook and, **per echelon**, trains a Random Forest to predict the part of demand ARIMA could not explain.

**Method (Zhang-style residual hybrid):**
- `Final_Hybrid = ARIMA_Pred + RF_residual_prediction`
- RF trains **only on `Split=='train'` rows** and is evaluated **only on `Split=='test'` rows** — clean separation, no leakage.
- **Three models are compared per echelon: ARIMA-only, Hybrid-resid (Zhang: RF predicts ARIMA's residual, added back), and Hybrid-augmented (ARIMA_Pred fed to RF as an extra feature).** A plain RF-only model is still *fitted internally* (it is part of the augmented hybrid's machinery) but is **excluded from the reported comparison** — the thesis question is whether the hybrid improves on ARIMA, not whether a black-box ML model can post a low error on its own. Metrics include pooled RMSE/MAE/MAPE plus robust **median per-series RMSE** and **scaled RMSE** so one large-volume product can't dominate.

**Pipeline:** Load residuals → prepare features → per-echelon {train RF on residuals, build hybrids} → ARIMA-vs-Hybrid comparison → comparison plots → feature importance → save results.

## Block 1 — Imports

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({'figure.dpi':130,'axes.grid':True,'grid.alpha':.3,'grid.linestyle':'--',
                     'axes.spines.top':False,'axes.spines.right':False})
PALETTE = ['#1B3A6B','#E8722A','#1E7B4A','#6B4C9A','#27847A','#C0392B']
print('Libraries ready.')

## Block 2 — Paths & load the ARIMA residual file

In [ ]:
BASE_DIR    = r'C:\Badhan Thesis'
RESULTS_DIR = os.path.join(BASE_DIR, 'ARIMA_Results')
FIGURES_DIR = os.path.join(BASE_DIR, 'RF_Figures')
RF_RESULTS  = os.path.join(BASE_DIR, 'RF_Results')
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RF_RESULTS, exist_ok=True)

RESID_FILE = os.path.join(RESULTS_DIR, 'arima_residuals_for_rf.xlsx')
if not os.path.exists(RESID_FILE) and os.path.exists('arima_residuals_for_rf.xlsx'):
    RESID_FILE = 'arima_residuals_for_rf.xlsx'

df = pd.read_excel(RESID_FILE)
print('Loaded:', RESID_FILE, '| shape:', df.shape)
print('Splits:'); print(df.groupby(['Echelon','Split']).size())
assert 'Split' in df.columns, "Residual file has no Split column. Re-run the UPDATED ARIMA notebook."

## Block 3 — Feature preparation

Features = calendar + festival + lag + rolling signals. `Season` (categorical) is one-hot encoded. The RF **target is `ARIMA_Residual`**. Lag/rolling NaNs (legitimate, early months) are filled with 0 so no rows are dropped.

In [ ]:
CAT_FEATURES = ['Season']
NUM_FEATURES = ['Month','Quarter','Festival_Spike_Level','Region_Festival_Boost',
                'Total_Festival_Score','Is_Festival_Month','Is_High_Festival',
                'Demand_Lag_1','Demand_Lag_2','Demand_Lag_3','Demand_Lag_12',
                'Rolling_Avg_3M','Rolling_Avg_6M','Rolling_Std_3M']
NUM_FEATURES = [c for c in NUM_FEATURES if c in df.columns]
CAT_FEATURES = [c for c in CAT_FEATURES if c in df.columns]

def make_xy(frame):
    X = frame[NUM_FEATURES].copy()
    for c in CAT_FEATURES:
        d = pd.get_dummies(frame[c].astype('object'), prefix=c)
        X = pd.concat([X, d], axis=1)
    X = X.fillna(0.0)
    return X

# Align dummy columns across train/test by building on the whole frame then splitting
X_all = make_xy(df)
FEATURE_COLS = X_all.columns.tolist()
print(f'{len(FEATURE_COLS)} features:', FEATURE_COLS)

## Block 4 — Per-echelon hybrid: train RF on residuals, evaluate the models

For each echelon: fit RF on train residuals; predict test residuals; build hybrid = ARIMA_Pred + RF_resid. A feature-augmented hybrid (ARIMA_Pred as an extra RF feature) is also built. All metrics on the **test** rows, original units. The reported comparison is **ARIMA vs Hybrid-resid vs Hybrid-augmented**.

In [ ]:
def metrics(actual, pred):
    actual, pred = np.asarray(actual,float), np.asarray(pred,float)
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    m = actual != 0
    mape = np.mean(np.abs((actual[m]-pred[m])/actual[m]))*100 if m.any() else np.nan
    return mae, rmse, mape

RF_PARAMS = dict(n_estimators=300, max_depth=8, min_samples_leaf=3,
                 max_features='sqrt', random_state=42, n_jobs=-1)

# Series identifier: lets us compute per-series (not just pooled) errors so one
# large-volume product cannot dominate an echelon's metric.
SERIES_KEYS = [c for c in ['Location','Product Name','Sales Channel'] if c in df.columns]

# Models reported in the thesis comparison (RF-only deliberately excluded).
MODELS = ['ARIMA','Hybrid-resid','Hybrid-augmented']

rows = []            # pooled metrics per (echelon, model)
per_series_rows = [] # per-series RMSE per (echelon, model) for median + scaled error
preds_store = {}
importances = {}

for ech in ['E1_Farm','E2_Distribution','E3_Retail']:
    sub = df[df['Echelon']==ech]
    tr = sub[sub['Split']=='train'].copy()
    te = sub[sub['Split']=='test'].copy()
    if len(tr) < 20 or len(te) < 5:
        print(f'{ech}: too few rows (train={len(tr)}, test={len(te)}) — skipped'); continue

    Xtr = make_xy(tr).reindex(columns=FEATURE_COLS, fill_value=0)
    Xte = make_xy(te).reindex(columns=FEATURE_COLS, fill_value=0)

    # --- Hybrid 1 (residual / Zhang): RF predicts the ARIMA RESIDUAL, add back ---
    rf_resid = RandomForestRegressor(**RF_PARAMS).fit(Xtr, tr['ARIMA_Residual'].values)
    resid_hat = rf_resid.predict(Xte)
    hybrid_resid_pred = te['ARIMA_Pred'].values + resid_hat

    # --- Hybrid 2 (feature-augmented): ARIMA_Pred is an EXTRA feature; RF predicts demand ---
    Xtr_aug = Xtr.copy(); Xtr_aug['ARIMA_Pred'] = tr['ARIMA_Pred'].values
    Xte_aug = Xte.copy(); Xte_aug['ARIMA_Pred'] = te['ARIMA_Pred'].values
    rf_aug = RandomForestRegressor(**RF_PARAMS).fit(Xtr_aug, tr['Actual'].values)
    hybrid_aug_pred = rf_aug.predict(Xte_aug)

    actual = te['Actual'].values
    arima_pred = te['ARIMA_Pred'].values

    # Only the three reported models (no RF-only in the comparison).
    model_preds = {'ARIMA':arima_pred,
                   'Hybrid-resid':hybrid_resid_pred,
                   'Hybrid-augmented':hybrid_aug_pred}

    # pooled metrics
    for model, pred in model_preds.items():
        mae, rmse, mape = metrics(actual, pred)
        rows.append({'Echelon':ech, 'Model':model, 'n_test':len(te),
                     'MAE':round(mae,2), 'RMSE':round(rmse,2), 'MAPE':round(mape,2)})

    # per-series RMSE + scaled error (RMSE divided by that series' mean actual)
    te_idx = te.reset_index(drop=True)
    grp = te_idx.groupby(SERIES_KEYS, dropna=False).indices
    for model, pred in model_preds.items():
        for gk, locs in grp.items():
            a = actual[locs]; p = np.asarray(pred)[locs]
            if len(a) == 0: continue
            srmse = np.sqrt(mean_squared_error(a, p))
            scale = np.mean(np.abs(a)) if np.mean(np.abs(a)) > 0 else np.nan
            per_series_rows.append({'Echelon':ech, 'Model':model,
                                    'series_RMSE':srmse,
                                    'scaled_RMSE':(srmse/scale if scale==scale else np.nan)})

    preds_store[ech] = {'actual':actual, **model_preds}
    importances[ech] = pd.Series(rf_aug.feature_importances_,
                                 index=Xtr_aug.columns).sort_values(ascending=False)
    print(f'{ech}: trained (train={len(tr)}, test={len(te)})')

results = pd.DataFrame(rows)
per_series = pd.DataFrame(per_series_rows)
print('\nDone. Models compared:', MODELS)

## Block 5 — Results table: ARIMA vs RF-only vs Hybrid-resid vs Hybrid-augmented

In [ ]:
# MODELS defined in Block 4 = ['ARIMA','Hybrid-resid','Hybrid-augmented'] (RF-only excluded)
# Pooled metrics; lower MAE/RMSE/MAPE = better
for metric in ['MAE','RMSE','MAPE']:
    piv = results.pivot(index='Echelon', columns='Model', values=metric)[MODELS]
    piv['Best'] = piv.idxmin(axis=1)
    print(f'\n=== {metric} by echelon — POOLED (lower is better) ===')
    display(piv.round(2))

# Robust view: MEDIAN per-series RMSE and MEDIAN scaled RMSE (one product can't dominate)
med_rmse = per_series.groupby(['Echelon','Model'])['series_RMSE'].median().unstack()[MODELS]
med_rmse['Best'] = med_rmse.idxmin(axis=1)
print('\n=== MEDIAN per-series RMSE (robust) ==='); display(med_rmse.round(2))

med_scaled = per_series.groupby(['Echelon','Model'])['scaled_RMSE'].median().unstack()[MODELS]
med_scaled['Best'] = med_scaled.idxmin(axis=1)
print('\n=== MEDIAN scaled RMSE (RMSE / series mean; dimensionless) ==='); display(med_scaled.round(3))

results.to_csv(os.path.join(RF_RESULTS,'hybrid_comparison.csv'), index=False)
per_series.to_csv(os.path.join(RF_RESULTS,'per_series_errors.csv'), index=False)

## Block 6 — Comparison plots

In [ ]:
# BAR: MEDIAN per-series error per model, grouped by echelon (robust to outliers)
fig, axes = plt.subplots(1,2, figsize=(15,5))
cols3 = [PALETTE[0],PALETTE[3],PALETTE[2]]   # ARIMA, Hybrid-resid, Hybrid-augmented
panels = [('series_RMSE','Median per-series RMSE'), ('scaled_RMSE','Median scaled RMSE')]
for ax,(col,title) in zip(axes, panels):
    piv = per_series.groupby(['Echelon','Model'])[col].median().unstack()[MODELS]
    piv = piv.reindex(['E1_Farm','E2_Distribution','E3_Retail'])
    x = np.arange(len(piv)); w = 0.25
    for j,(mname,c) in enumerate(zip(MODELS, cols3)):
        ax.bar(x+(j-1)*w, piv[mname].values, w, label=mname, color=c, edgecolor='white')
    ax.set_xticks(x); ax.set_xticklabels(['E1','E2','E3']); ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=8)
fig.suptitle('Model comparison by echelon — ARIMA vs Hybrids (lower is better)', fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(os.path.join(FIGURES_DIR,'rf1_model_comparison.png'), dpi=150); plt.show()

In [ ]:
# LINE: actual vs models, pooled test points per echelon (ordered for readability)
fig, axes = plt.subplots(3,1, figsize=(14,11))
for ax, ech in zip(axes, ['E1_Farm','E2_Distribution','E3_Retail']):
    if ech not in preds_store: continue
    d = preds_store[ech]
    order = np.argsort(d['actual'])
    xx = np.arange(len(d['actual']))
    ax.plot(xx, d['actual'][order], color='black', lw=2, label='Actual', marker='o', ms=3)
    ax.plot(xx, d['ARIMA'][order], color=PALETTE[1], lw=1.4, ls=':', label='ARIMA', alpha=.85)
    ax.plot(xx, d['Hybrid-resid'][order], color=PALETTE[3], lw=1.4, ls='--', label='Hybrid-resid', alpha=.85)
    ax.plot(xx, d['Hybrid-augmented'][order], color=PALETTE[2], lw=1.9, label='Hybrid-augmented')
    # keep y-axis readable even if a residual hybrid still overshoots
    hi = np.nanpercentile(d['actual'], 99) * 1.5
    ax.set_ylim(0, max(hi, np.nanpercentile(d['actual'],100)*1.05))
    ax.set_title(f'{ech} — test points (sorted by actual)', fontweight='bold')
    ax.set_ylabel('Quantity'); ax.legend(loc='upper left', fontsize=8)
fig.suptitle('Actual vs ARIMA vs Hybrids (test set)', fontweight='bold', y=1.0)
plt.tight_layout(); plt.savefig(os.path.join(FIGURES_DIR,'rf2_actual_vs_models.png'), dpi=150); plt.show()

## Block 7 — Feature importance (what RF used to model the residuals)

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(17,5))
for ax,(ech,c) in zip(axes, [('E1_Farm',PALETTE[0]),('E2_Distribution',PALETTE[1]),('E3_Retail',PALETTE[2])]):
    if ech not in importances: continue
    top = importances[ech].head(10)[::-1]
    ax.barh(top.index, top.values, color=c, edgecolor='white')
    ax.set_title(f'{ech} — top RF features', fontweight='bold'); ax.set_xlabel('Importance')
fig.suptitle('Feature importance — augmented hybrid RF (incl. ARIMA_Pred)', fontweight='bold', y=1.05)
plt.tight_layout(); plt.savefig(os.path.join(FIGURES_DIR,'rf3_feature_importance.png'), dpi=150); plt.show()

## Block 8 — Save results & summary

In [ ]:
results.to_excel(os.path.join(RF_RESULTS,'hybrid_results.xlsx'), index=False)
print('Saved hybrid results to', os.path.join(RF_RESULTS,'hybrid_results.xlsx'))

print('\n=== SUMMARY: best model per echelon (by MEDIAN per-series RMSE) ===')
med = per_series.groupby(['Echelon','Model'])['series_RMSE'].median().unstack()[MODELS]
for ech in ['E1_Farm','E2_Distribution','E3_Retail']:
    if ech not in med.index: continue
    row = med.loc[ech]
    best = row.idxmin()
    arima_v = row['ARIMA']; haug = row['Hybrid-augmented']; hres = row['Hybrid-resid']
    print(f'  {ech:<16} winner={best:<17} | ARIMA={arima_v:>8.1f}  '
          f'H-resid={hres:>8.1f}  H-aug={haug:>8.1f}')
    # did the BEST hybrid beat plain ARIMA?
    best_hybrid = min(haug, hres)
    tag = 'beats' if best_hybrid < arima_v else 'does NOT beat'
    print(f'    -> best hybrid {tag} ARIMA ({(arima_v-best_hybrid)/arima_v*100:+.1f}% vs ARIMA)')

## Block 9 — Next step: multi-echelon linkage (thesis novelty)

The models above forecast each echelon **independently**. The thesis contribution is to test whether **sharing information across echelons** (e.g. using an upstream forecast as a downstream feature) reduces error and dampens the bullwhip effect. That is the next notebook: add the upstream echelon's forecast as a feature to the downstream RF and re-run this comparison.